In [2]:
%load_ext rpy2.ipython
%load_ext autoreload
%autoreload 2

Error importing in API mode: ImportError("dlopen(/opt/anaconda3/envs/fd_library/lib/python3.12/site-packages/_rinterface_cffi_api.abi3.so, 0x0002): symbol not found in flat namespace '_R_BaseEnv'")
Trying to import in ABI mode.


In [3]:
import pandas as pd

from utils import functional_richness, functional_evenness, functional_divergence
from utils import euclidean_distance

## Testing on a small dataset

In [4]:
traits = pd.DataFrame(
    [[1, 2], [2, 3], [3, 1], [4, 2]],
    columns=["Trait_1", "Trait_2"],
    index=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
)

abundances = pd.DataFrame(
    [[5, 3, 2, 1], [1, 2, 0, 2]],
    columns=["Sp_0", "Sp_1", "Sp_2", "Sp_3"],
    index=["Plot_A", "Plot_B"],
)

In [5]:
FRic = functional_richness(
    abundances, traits, relative_abundance=False, standardize_traits=True
)

distance_matrix_euclidean = euclidean_distance(
    traits, metric="euclidean", standardize=True
)
FEve = functional_evenness(
    abundances, distance_matrix_euclidean, relative_abundance=False
)

FDiv = functional_divergence(
    abundances, traits, relative_abundance=False, standardize_traits=True
)

python_results_df = FRic.merge(FEve, on="PID").merge(FDiv, on="PID")
display(python_results_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence
0,Plot_A,3.794733,0.734321,0.947928
1,Plot_B,1.897367,0.989082,0.846568


In [ ]:
r_results_df = None

In [6]:
%%R -i traits,abundances -o r_results_df
library(FD)


trait_mat <- as.matrix(traits)
abun_mat <- as.matrix(abundances)

res <- dbFD(x = trait_mat, a = abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE)

r_results_df <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv
)

FRic: No dimensionality reduction was required. The 2 PCoA axes were kept as 'traits'. 


Loading required package: ade4
Loading required package: ape
Loading required package: geometry
Loading required package: vegan
Loading required package: permute


In [7]:
display(r_results_df)

,PID,R_FRic,R_FEve,R_FDiv
Plot_A,Plot_A,2.846050,0.734321,0.947928
Plot_B,Plot_B,1.423025,0.989082,0.846568


## Testing on the birds dataset

Load the data

In [8]:
bird_loc = pd.read_csv("./data/example/bird/bird_location.csv")
bird_traits = pd.read_csv("./data/example/bird/bird_traits.csv")

bird_loc = bird_loc.set_index("PID")
bird_traits = bird_traits.set_index("Species")

In [9]:
FRic_bird = functional_richness(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits=True
)

distance_matrix_euclidean_bird = euclidean_distance(
    bird_traits, metric="euclidean", standardize=True
)
FEve_bird = functional_evenness(
    bird_loc,
    distance_matrix_euclidean_bird,
    relative_abundance=False,
    abundance_weighted=True,
)

FDiv_bird = functional_divergence(
    bird_loc, bird_traits, relative_abundance=False, standardize_traits=True
)

python_results_df_bird = FRic_bird.merge(FEve_bird, on="PID").merge(FDiv_bird, on="PID")
display(python_results_df_bird)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence
0,elev_250,66.661794,0.656412,0.747405
1,elev_500,72.128929,0.651052,0.755107
2,elev_1000,43.756363,0.623858,0.743327
3,elev_1500,25.703033,0.568285,0.742684
4,elev_2000,7.797544,0.605025,0.730325
5,elev_2500,7.111826,0.631438,0.703676
6,elev_3000,6.812401,0.616293,0.700852
7,elev_3500,1.441212,0.592614,0.671813


In [17]:
r_results_df_bird = None

In [18]:
%%R -i bird_loc,bird_traits -o r_results_df_bird,r_pcoa_coordinates
library(FD)
bird_trait_mat <- as.matrix(bird_traits)
bird_abun_mat <- as.matrix(bird_loc)

res <- dbFD(x = bird_trait_mat, a = bird_abun_mat, calc.FRic = TRUE, calc.FDiv = TRUE, print.pco = TRUE)

r_results_df_bird <- data.frame(
    PID = names(res$FEve), 
    R_FRic = res$FRic,
    R_FEve = res$FEve,
    R_FDiv = res$FDiv
)

r_pcoa_coordinates <- res$x.axes

FRic: No dimensionality reduction was required. All 4 PCoA axes were kept as 'traits'. 


In [19]:
merge_df = python_results_df_bird.merge(r_results_df_bird, on="PID")

In [20]:
merge_df["FRic_ratio"] = merge_df["Functional_Richness"] / merge_df["R_FRic"]
merge_df["FEve_ratio"] = merge_df["Functional_Evenness"] / merge_df["R_FEve"]
merge_df["FDiv_ratio"] = merge_df["Functional_Divergence"] / merge_df["R_FDiv"]
display(merge_df)

,PID,Functional_Richness,Functional_Evenness,Functional_Divergence,R_FRic,R_FEve,R_FDiv,FRic_ratio,FEve_ratio,FDiv_ratio
0,elev_250,66.661794,0.656412,0.747405,66.048816,0.656412,0.747405,1.009281,1.0,1.0
1,elev_500,72.128929,0.651052,0.755107,71.465678,0.651052,0.755107,1.009281,1.0,1.0
2,elev_1000,43.756363,0.623858,0.743327,43.354008,0.623858,0.743327,1.009281,1.0,1.0
3,elev_1500,25.703033,0.568285,0.742684,25.466685,0.568285,0.742684,1.009281,1.0,1.0
4,elev_2000,7.797544,0.605025,0.730325,7.725843,0.605025,0.730325,1.009281,1.0,1.0
5,elev_2500,7.111826,0.631438,0.703676,7.046431,0.631438,0.703676,1.009281,1.0,1.0
6,elev_3000,6.812401,0.616293,0.700852,6.749758,0.616293,0.700852,1.009281,1.0,1.0
7,elev_3500,1.441212,0.592614,0.671813,1.427960,0.592614,0.671813,1.009281,1.0,1.0
